In [ ]:
!pip install kafka-python
!pip install google-api-python-client
!pip install boto3
!pip install mysql-connector-python


In [ ]:
!pip install kafka-python isodate requests



# Producer Code

In [ ]:
import json
import requests
from kafka import KafkaProducer
import time

# YouTube API Setup
API_KEY = ""
YOUTUBE_PLAYLIST_ITEMS_URL = "https://www.googleapis.com/youtube/v3/playlistItems"
YOUTUBE_VIDEO_URL = "https://www.googleapis.com/youtube/v3/videos"
YOUTUBE_CHANNEL_URL = "https://www.googleapis.com/youtube/v3/channels"

# List of Indian Food YouTube Channel IDs
CHANNEL_IDS = [
    "UCnwL537dF3kV8hGVHvvof3Q",  # Golgappa Girl
    "UClfidxoNRvDO361TM5rw64Q",  # Cook with Parul
    "UCChqsCRFePrP2X897iQkyAA",  # Kabita's Kitchen
    "UCmoX4QULJ9MB00xW4coMiOw",  # Sanjeev Kappor Khazana
    "UCHGktfcQq2BY_8tGPHwvm7g",  # Madras Samayal
    "UCPPIsrNlEkaFQBk-4uNkOaw",  # Hebbars Kitchen
    "UCVC9JRU1JpfupKHVV0sThOg"   # Veggie Paaji
]

# Kafka Producer Setup
producer = KafkaProducer(
    bootstrap_servers='',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

def get_uploads_playlist_id(channel_id):
    params = {
        'part': 'contentDetails',
        'id': channel_id,
        'key': API_KEY
    }
    response = requests.get(YOUTUBE_CHANNEL_URL, params=params).json()
    return response['items'][0]['contentDetails']['relatedPlaylists']['uploads']

def fetch_all_video_ids(playlist_id):
    video_ids = []
    next_page_token = None

    while True:
        params = {
            'part': 'contentDetails',
            'playlistId': playlist_id,
            'maxResults': 50,
            'pageToken': next_page_token,
            'key': API_KEY
        }
        response = requests.get(YOUTUBE_PLAYLIST_ITEMS_URL, params=params).json()
        for item in response.get('items', []):
            video_ids.append(item['contentDetails']['videoId'])

        next_page_token = response.get('nextPageToken')
        if not next_page_token:
            break
        time.sleep(0.1)  # Respect API rate limits
    return video_ids

def fetch_video_details(video_ids):
    videos = []
    for i in range(0, len(video_ids), 50):
        batch = video_ids[i:i+50]
        params = {
            'part': 'snippet,statistics',
            'id': ','.join(batch),
            'key': API_KEY
        }
        response = requests.get(YOUTUBE_VIDEO_URL, params=params).json()
        for video in response.get('items', []):
            title = video['snippet']['title']
            if "shorts" not in title.lower():  # Filter out Shorts
                videos.append({
                    'videoId': video['id'],
                    'title': title,
                    'publishedAt': video['snippet']['publishedAt'],
                    'viewCount': video['statistics'].get('viewCount'),
                    'likeCount': video['statistics'].get('likeCount'),
                    'channelTitle': video['snippet']['channelTitle']
                })
    return videos

# Main Execution Loop
for channel_id in CHANNEL_IDS:
    try:
        print(f"📺 Processing channel: {channel_id}")
        playlist_id = get_uploads_playlist_id(channel_id)
        video_ids = fetch_all_video_ids(playlist_id)
        videos = fetch_video_details(video_ids)
        print(f"🚀 Sending {len(videos)} videos for {channel_id}")

        for video in videos:
            producer.send('youtube-topic', value=video)
            #print("✅ Sent:", video['title'])

        time.sleep(0.5)  # Avoid hitting API too fast
    except Exception as e:
        print(f"❌ Error processing channel {channel_id}:", e)

producer.flush()
producer.close()


📺 Processing channel: UCnwL537dF3kV8hGVHvvof3Q
🚀 Sending 307 videos for UCnwL537dF3kV8hGVHvvof3Q
📺 Processing channel: UClfidxoNRvDO361TM5rw64Q
🚀 Sending 2354 videos for UClfidxoNRvDO361TM5rw64Q
📺 Processing channel: UCChqsCRFePrP2X897iQkyAA
🚀 Sending 1726 videos for UCChqsCRFePrP2X897iQkyAA
📺 Processing channel: UCmoX4QULJ9MB00xW4coMiOw
🚀 Sending 13340 videos for UCmoX4QULJ9MB00xW4coMiOw
📺 Processing channel: UCHGktfcQq2BY_8tGPHwvm7g
🚀 Sending 912 videos for UCHGktfcQq2BY_8tGPHwvm7g
📺 Processing channel: UCPPIsrNlEkaFQBk-4uNkOaw
🚀 Sending 2886 videos for UCPPIsrNlEkaFQBk-4uNkOaw
📺 Processing channel: UCVC9JRU1JpfupKHVV0sThOg
🚀 Sending 1332 videos for UCVC9JRU1JpfupKHVV0sThOg
